# PREPROCESSING AND FEATURE ENGINEERING

In [ ]:

from pyspark.sql.functions import (
    col, when, hour, dayofweek, month, year,
    to_timestamp, isnan, count, lit, udf
)
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, Imputer
)
from pyspark.ml import Pipeline

In [ ]:
# Drop columns not useful for ML

drop_cols = ['id', 'case_number', 'block', 'updated_on', 'location',
             'x_coordinate', 'y_coordinate', 'description']

df = df.drop(*drop_cols)

print("Columns after dropping irrelevant features:")
print(df.columns)

Columns after dropping irrelevant features:
['date', 'iucr', 'primary_type', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'latitude', 'longitude', 'year']


In [ ]:

#  Parse date and extract temporal features

df = df.withColumn("date_parsed", to_timestamp(col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS"))

df = df \
    .withColumn("hour_of_day",   hour(col("date_parsed"))) \
    .withColumn("day_of_week",   dayofweek(col("date_parsed"))) \
    .withColumn("month_of_year", month(col("date_parsed")))

# Drop original date columns
df = df.drop("date", "date_parsed")

print("\nTemporal features added: hour_of_day, day_of_week, month_of_year")


Temporal features added: hour_of_day, day_of_week, month_of_year


In [ ]:
#  Handle missing values

# Fill missing categorical with 'UNKNOWN'
cat_fill = ['location_description', 'fbi_code']
for c in cat_fill:
    df = df.withColumn(c, when(col(c).isNull(), "UNKNOWN").otherwise(col(c)))

In [ ]:
# Fill missing numerical with median using Imputer
num_fill_cols = ['district', 'ward', 'community_area', 'latitude', 'longitude']
imputer = Imputer(
    inputCols=num_fill_cols,
    outputCols=num_fill_cols,
    strategy="median"
)
df = imputer.fit(df).transform(df)

print("\nMissing values handled")
print("Categorical : filled with UNKNOWN")
print("Numerical   : filled with median via Imputer")


Missing values handled
Categorical : filled with UNKNOWN
Numerical   : filled with median via Imputer


In [ ]:
#  Convert boolean columns to integer

df = df \
    .withColumn("arrest",  col("arrest").cast("integer")) \
    .withColumn("domestic", col("domestic").cast("integer"))

print("\nBoolean columns cast to integer")
print("arrest  : 0 = No Arrest, 1 = Arrest")
print("domestic: 0 = Non-domestic, 1 = Domestic")


Boolean columns cast to integer
arrest  : 0 = No Arrest, 1 = Arrest
domestic: 0 = Non-domestic, 1 = Domestic


In [ ]:
#  Encode categorical columns

cat_cols = ['primary_type', 'location_description', 'fbi_code', 'iucr']

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in cat_cols
]

encoders = [
    OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_enc")
    for c in cat_cols
]

print("\nCategorical columns to be encoded:")
for c in cat_cols:
    print(f"  {c}")


Categorical columns to be encoded:
  primary_type
  location_description
  fbi_code
  iucr


In [ ]:

# Assemble feature vector
numeric_features = [
    'beat', 'district', 'ward', 'community_area',
    'latitude', 'longitude', 'hour_of_day',
    'day_of_week', 'month_of_year', 'domestic', 'year'
]

encoded_features = [c + "_enc" for c in cat_cols]
all_features     = numeric_features + encoded_features

assembler = VectorAssembler(
    inputCols=all_features,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

In [ ]:

# 2.7 - Build and run the preprocessing pipeline

pipeline_stages = indexers + encoders + [assembler, scaler]

preprocessing_pipeline = Pipeline(stages=pipeline_stages)

print("\nFitting preprocessing pipeline")
pipeline_model = preprocessing_pipeline.fit(df)
df_processed   = pipeline_model.transform(df)

# Keep only what is needed for ML
df_ml = df_processed.select("features", "arrest")

print("Pipeline complete")
print("Final ML DataFrame columns:", df_ml.columns)
print("Rows in processed data     :", df_ml.count())


Fitting preprocessing pipeline
Pipeline complete
Final ML DataFrame columns: ['features', 'arrest']
Rows in processed data     : 4459898


In [ ]:

#  Persist processed data (memory management)

df_ml.persist()
df.unpersist()

print("\ndf_ml persisted to memory")
print("Original df unpersisted to free memory")


df_ml persisted to memory
Original df unpersisted to free memory


In [ ]:

# Save preprocessed data for reuse

PROCESSED_PATH = "/content/chicago_crimes/processed"

df_processed.write \
    .mode("overwrite") \
    .parquet(PROCESSED_PATH)

print("\nProcessed data saved to:", PROCESSED_PATH)


Processed data saved to: /content/chicago_crimes/processed


In [ ]:
# Save file for Tableau Dashboard 1


TABLEAU_DIR = "/content/chicago_crimes/tableau"
os.makedirs(TABLEAU_DIR, exist_ok=True)

# Missing value summary for Tableau
missing_summary = []
for c in df.columns:
    null_count = df.filter(col(c).isNull()).count()
    missing_summary.append({
        "column"       : c,
        "null_count"   : null_count,
        "null_percent" : round(null_count / df.count() * 100, 2)
    })

pd.DataFrame(missing_summary).to_csv(
    f"{TABLEAU_DIR}/tableau_data_quality.csv", index=False
)

# Year-wise record count for pipeline monitoring
year_counts = df_processed \
    .groupBy("year") \
    .agg(count("*").alias("record_count")) \
    .orderBy("year") \
    .toPandas()

year_counts.to_csv(
    f"{TABLEAU_DIR}/tableau_year_distribution.csv", index=False
)

# Crime type distribution
crime_dist = df_processed \
    .groupBy("primary_type", "year") \
    .agg(count("*").alias("count")) \
    .orderBy("year", "count") \
    .toPandas()

crime_dist.to_csv(
    f"{TABLEAU_DIR}/tableau_crime_distribution.csv", index=False
)

print("\nTableau Dashboard 1 files saved:")
print(f"  {TABLEAU_DIR}/tableau_data_quality.csv")
print(f"  {TABLEAU_DIR}/tableau_year_distribution.csv")
print(f"  {TABLEAU_DIR}/tableau_crime_distribution.csv")


Tableau Dashboard 1 files saved:
  /content/chicago_crimes/tableau/tableau_data_quality.csv
  /content/chicago_crimes/tableau/tableau_year_distribution.csv
  /content/chicago_crimes/tableau/tableau_crime_distribution.csv
